# Vector stores and semantic search



In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch

/home/david/computational-intelligence/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

## Part I: Basic vector store implementation

In [3]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        
        embebido = self.embedding_model.encode(
            [str(document.text) for document in documents], # nininini no soy de tipado fuerte pero no me pasen un número porque no tengo idea de qué hacer con él en un parámetro de string ~ atentamente el lenguaje preferido de aparentemente todos
            convert_to_tensor=True
        )
        
        self.embeddings = embebido if self.embeddings is None else torch.cat((self.embeddings, embebido),dim=0) # yo cuando 🕯️🐱

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if len(self.documents) == 0:
            return []
        
        valores, ubicaciones = torch.topk(
            torch.nn.functional.normalize(self.embeddings,p=2)
            @
            torch.nn.functional.normalize(self.embedding_model.encode(query,convert_to_tensor=True),p=2,dim=0),
            min(top_k, len(self.documents))
        )
        
        return [
            SearchResult(float(v), self.documents[int(u)]) # nininini no soy de tipado fuerte pero no me pasen un Tensor unidimensional porque no tengo idea de qué hacer con él en un parámetro escalar o en un accesador de lista porque tampoco me da cabeza para convertirlo a entero ~ atentamente el lenguaje preferido de aparentemente todos
            for v, u in zip(valores, ubicaciones)
        ]

## Part II: Filtering by metadata

In [4]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass

# Cargar dataset:

In [5]:
animal_df = pd.read_csv('./datasets/animal-fun-facts-dataset.csv')

animal_df

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark
...,...,...,...,...,...
7729,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,"Like many reptiles, the incubation temperature...",NaN,/wiki/Spilotes_pullatus
7730,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,"The yellow rat snake, or chicken snake, is kno...",NaN,/wiki/Spilotes_pullatus
7731,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,"Like pythons and boas, rat snakes are constric...",NaN,/wiki/Spilotes_pullatus
7732,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,Yellow rat snakes spend much time underground ...,NaN,/wiki/Spilotes_pullatus


In [6]:
anilist = animal_df.values.tolist()

animal_docs = []

for ani in anilist:
    animal_docs.append(Document(text=ani[2], metadata={
        'animal_name': ani[0],
        'source': ani[1],
        'media_link': ani[3],
        'wikipedia_link': ani[4]
    }))
    
(len(animal_docs), animal_docs[-1].text, animal_docs[-1].metadata)

(7734,
 'Truly arboreal, yellow rat snakes will commonly climb trees to reach and devour birds and their eggs. The snake is known to climb to heights of 60 feet search for prey in trees.',
 {'animal_name': 'yellow rat snake',
  'source': 'https://seaworld.org/animals/facts/reptiles/yellow-rat-snake/',
  'media_link': nan,
  'wikipedia_link': '/wiki/Spilotes_pullatus'})

In [7]:
animal_planet = VectorStore(embedding_model=SentenceTransformer("all-MiniLM-L6-v2", device=device.type))

animal_planet.add_documents(animal_docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3905.17it/s]


In [15]:
def mostrar_resultados(results: list[SearchResult]):
    return pd.DataFrame([
        {
            "score": r.score,
            "text": r.document.text,
            **r.document.metadata
        }
        for r in results
    ])

In [16]:
hey_google_whose_cat_is_fatter = animal_planet.search("biggest feline")

mostrar_resultados(hey_google_whose_cat_is_fatter)

,score,text,animal_name,source,media_link,wikipedia_link
0,0.907390,The largest feline in the world!,tiger,https://a-z-animals.com/animals/tiger/,NaN,/wiki/Tiger
1,0.779930,The largest feline on the American continent!,jaguar,https://a-z-animals.com/animals/jaguar/,NaN,/wiki/Jaguar
2,0.771067,The Second Largest feline in North America,cougar,https://a-z-animals.com/animals/cougar/,NaN,/wiki/Cougar
3,0.650733,The largest animal on Earth,blue whale,https://a-z-animals.com/animals/blue-whale/,NaN,/wiki/Blue_whale
4,0.633959,"The tiger is the largest of the four ""Big Cats""",tiger,https://www.animalfactsencyclopedia.com/Tiger-...,NaN,/wiki/Tiger


In [17]:
hey_google_what_color_is_a_blue_whale = animal_planet.search("blue whale color")

mostrar_resultados(hey_google_what_color_is_a_blue_whale)

,score,text,animal_name,source,media_link,wikipedia_link
0,0.687977,Blue whales belong to the cetacean suborder My...,blue whale,https://seaworld.org/animals/facts/mammals/blu...,NaN,/wiki/Blue_whale
1,0.666253,A Blue whale calf is anything but a little bab...,blue whale,https://factanimal.com/blue-whale/,NaN,/wiki/Blue_whale
2,0.634829,Actions displayed by blue whales appear that t...,blue whale,https://factanimal.com/blue-whale/,NaN,/wiki/Blue_whale
3,0.629402,The distinctive coloration of killer whales is...,killer whale,https://seaworld.org/animals/facts/mammals/kil...,NaN,/wiki/Orca
4,0.625435,The blue whale resembles a well-known sea craf...,blue whale,https://factanimal.com/blue-whale/,NaN,/wiki/Blue_whale


In [18]:
hey_google_is_python_faster_than_swift = animal_planet.search("python swift bird speed")

mostrar_resultados(hey_google_is_python_faster_than_swift)

,score,text,animal_name,source,media_link,wikipedia_link
0,0.536178,The bottle nose dolphins top speed is almost 3...,dolphin,https://www.animalfactsencyclopedia.com/Dolphi...,NaN,/wiki/Dolphin
1,0.535014,They are the fastest bird in the world.\nThe p...,peregrine falcon,https://factanimal.com/peregrine-falcon/,NaN,/wiki/Peregrine_falcon
2,0.510858,The shoebill’s flapping when flying is one of ...,shoebill,https://factanimal.com/shoebill/,NaN,/wiki/Shoebill
3,0.485257,Fastest animal on Earth,peregrine falcon,https://a-z-animals.com/animals/peregrine-falcon/,NaN,/wiki/Peregrine_falcon
4,0.476917,Commerson's dolphins are fast and highly maneu...,commerson's dolphin,https://seaworld.org/animals/facts/mammals/com...,NaN,/wiki/Commerson%27s_dolphin


In [19]:
hey_google_can_cows_sleep_sideways = animal_planet.search("cows sleeping sideways")

mostrar_resultados(hey_google_can_cows_sleep_sideways)

,score,text,animal_name,source,media_link,wikipedia_link
0,0.551949,Camels can sleep standing up..\nWhile they usu...,camel,https://factanimal.com/camel/,NaN,/wiki/Camel
1,0.549890,Cows have best friends.,cows,/r/AskReddit/comments/2evjpq/what_are_some_ani...,NaN,/wiki/Cattle
2,0.526776,Sloths sleep hanging completely upside-down,sloth,https://www.animalfactsencyclopedia.com/Sloth-...,NaN,/wiki/Sloth
3,0.513330,Cows produce more milk while listening to peac...,cow,/r/AskReddit/comments/gbh7zz/what_are_some_rea...,NaN,/wiki/Cattle
4,0.494758,Elephants can sleep standing up and only need ...,elephant,https://factanimal.com/elephants/,NaN,/wiki/Elephant


In [20]:
hey_google_do_cats_fear_fire_instinctively = animal_planet.search("cats fearing fire") # mi gato Totito se asustó por ver un asador encendido, siendo que NUNCA antes había estado expuesto a la imagen ni el calor de una llama

mostrar_resultados(hey_google_do_cats_fear_fire_instinctively)

,score,text,animal_name,source,media_link,wikipedia_link
0,0.535761,A group of wild cats is called a destruction,european wildcat,https://a-z-animals.com/animals/european-wildcat/,NaN,/wiki/European_wildcat
1,0.532066,Jaguars are the least likely of all the big ca...,jaguar,https://www.animalfactsencyclopedia.com/Jaguar...,NaN,/wiki/Jaguar
2,0.529389,They love bush fire.\nDuring bushfires and bur...,marabou stork,https://factanimal.com/marabou-stork/,NaN,/wiki/Marabou_stork
3,0.503653,Cat culls offer promising returns.\nThere are ...,numbat,https://factanimal.com/numbat/,NaN,/wiki/Numbat
4,0.503280,"Unlike the other ""big cats"" they cannot roar",snow leopard,https://www.animalfactsencyclopedia.com/Snow-l...,NaN,/wiki/Snow_leopard
